<p align="left">
  <img src="https://i.creativecommons.org/l/by/4.0/88x31.png" align="left">
</p>

Text provided under a Creative Commons Attribution license, CC-BY.  
All code is made available under the FSF-approved MIT license.  
(c) Kyle T. Mandli

In [2]:
from __future__ import print_function
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

# Persamaan Parabolik

Sekarang kita akhirnya beralih untuk menggabungkan apa yang telah kita pelajari tentang diskretisasi ruang pada *boundary value problem* dengan diskretisasi waktu pada *initial value problem*. Jenis persamaan pertama yang akan kita pelajari adalah persamaan parabolik, di mana persamaan panas (*heat equation*)

$$
u_t = \kappa u_{xx}
$$

akan menjadi fokus kita. Banyak sifat umum dari metode numerik yang akan kita turunkan untuk persamaan panas ini juga berlaku untuk banyak persamaan parabolik lainnya.

Selain persamaan di atas, kita juga memerlukan kondisi batas:

$$
u(0, t) = g_0(t), \quad t > 0
$$
$$
u(1, t) = g_1(t), \quad t > 0
$$

serta kondisi awal:

$$
u(x, 0) = u_0(x)
$$

Di sini kita mengasumsikan domain $\Omega = [0, 1]$, $t_0 = 0$, dan bahwa kita menggunakan kondisi batas Dirichlet (kita akan melihat implementasi lain nanti). Kita juga akan mendiskretisasi domain ruang-waktu sehingga:

$$
x_i = i\Delta x, \quad t_n = n\Delta t
$$

dan bahwa pendekatan numerik $U$ akan mendekati fungsi sebenarnya $u$ pada titik $(x_i, t_n)$ sehingga:

$$
U_i^n \approx u(x_i, t_n)
$$

Sebagai percobaan pertama untuk mendiskretisasi persamaan panas, mari kita gunakan metode Forward Euler untuk waktu dan beda hingga terpusat orde dua untuk ruang sehingga diperoleh:

$$
\frac{U_i^{n+1} - U_i^n}{\Delta t} = \frac{1}{\Delta x^2}(U_{i-1}^n - 2U_i^n + U_{i+1}^n)
$$

atau dalam bentuk pembaruan:

$$
U_i^{n+1} = U_i^n + \frac{\Delta t}{\Delta x^2}(U_{i-1}^n - 2U_i^n + U_{i+1}^n)
$$

Salah satu cara melihat diskretisasi ini adalah dengan menganggapnya sebagai *initial value problem* sehingga:

$$
U_i^{n+1} = U_i^n + \Delta t \, f(t_n, U_i^n)
$$

di mana sekarang diskretisasi ruang terdapat dalam fungsi $f$ sehingga pada kasus di atas kita memiliki:

$$
f(t_n, U_i^n) = \frac{1}{\Delta x^2}(U_{i-1}^n - 2U_i^n + U_{i+1}^n)
$$

yang memberi kita cara untuk menganalisis metode ini dalam konteks metode numerik untuk *initial value problem*.

Metode lain yang juga menggunakan satu langkah tetapi lebih praktis (seperti yang akan kita lihat) disebut metode *Crank–Nicholson*. Metode ini didasarkan pada diskretisasi metode trapesium sehingga:

$$
\frac{U_i^{n+1} - U_i^n}{\Delta t} = \frac{1}{2}(f(U_i^n) + f(U_i^{n+1}))
$$

$$
= \frac{1}{2} \frac{U_{i-1}^n - 2U_i^n + U_{i+1}^n + U_{i-1}^{n+1} - 2U_i^{n+1} + U_{i+1}^{n+1}}{\Delta x^2}
$$

$$
\Rightarrow
$$

$$
U_i^{n+1} = U_i^n + \frac{\Delta t}{2\Delta x^2}(U_{i-1}^n - 2U_i^n + U_{i+1}^n + U_{i-1}^{n+1} - 2U_i^{n+1} + U_{i+1}^{n+1})
$$

Cobalah menggambar *stencil* dari metode ini.

Dari rumus pembaruan, kita dapat melakukan sedikit aljabar untuk memperoleh:

$$
-r U_{i-1}^{n+1} + (1 + 2r) U_i^{n+1} - r U_{i+1}^{n+1}
= r U_{i-1}^n + (1 - 2r) U_i^n + r U_{i+1}^n
$$

dengan:

$$
r = \frac{\Delta t}{2\Delta x^2}
$$

Cobalah menuliskan sistem persamaan ini jika kita memiliki $m = 5$.

Ini membentuk sistem persamaan tridiagonal berbentuk:

$$
A U^{n+1} = f(t_n, U^n)
$$

di mana:

$$
A =
\begin{bmatrix}
1+2r & -r \\
-r & 1+2r & -r \\
& -r & 1+2r & -r \\
& & \ddots & \ddots & \ddots \\
& & & -r & 1+2r & -r \\
& & & & -r & 1+2r
\end{bmatrix}
$$

dan

$$
f(t_n, U^n) =
\begin{bmatrix}
r(g_0(t_n) + g_0(t_{n+1})) + (1 - 2r)U_1^n + rU_2^n \\
rU_1^n + (1 - 2r)U_2^n + rU_3^n \\
rU_2^n + (1 - 2r)U_3^n + rU_4^n \\
\vdots \\
rU_{m-2}^n + (1 - 2r)U_{m-1}^n + rU_m^n \\
rU_{m-1}^n + (1 - 2r)U_m^n + r(g_1(t_n) + g_1(t_{n+1}))
\end{bmatrix}
$$

Dari pembahasan kita tentang metode iteratif, kita mungkin dapat menyelesaikan sistem ini dalam $\mathcal{O}(m)$ langkah, sehingga metode Crank–Nicholson untuk persamaan panas menjadi sama efisiennya dengan metode eksplisit di atas. Keuntungan dari metode implisit adalah memungkinkan penggunaan langkah waktu yang jauh lebih besar dibanding metode eksplisit karena kendala stabilitas, yang akan kita bahas lebih lanjut (persamaan panas dapat dianggap sebagai sistem ODE yang *stiff*).

---

## 🔸 Catatan Tambahan (biar lebih paham)
- Persamaan panas → menggambarkan penyebaran suhu dari daerah panas ke dingin  
- Metode eksplisit → mudah tapi bisa tidak stabil kalau $\Delta t$ terlalu besar  
- Metode Crank–Nicholson → lebih stabil karena pakai rata-rata dua waktu  
- Matriks tridiagonal → bisa diselesaikan cepat (ini penting di komputasi)